## **Profile LLM Inference - with optimizations**

In [ ]:
!pip install --upgrade transformers accelerate

#### Deploy an LLM and make inference with nsys profiling enabled. Review the report and optimize the below code to profile optimized inference. Compare both the reports to observe the difference and bottlenecks.
Current notebook is to generate a report with some optimizations.
We are loading a half precision model (BF16), and the system prompt, model inference parameters have been updated. 

**Key Metrics to Observe** - Inference Latency, GPU Utilization and Memory Utilization

In [ ]:
import torch
import nvtx
from transformers import pipeline

torch.cuda.empty_cache()

@nvtx.annotate()
def invoke_model(query):
    nvtx.push_range("total execution trace", domain="nvtx-inference")
   
    nvtx.push_range("load model", domain="nvtx-inference")
    pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", dtype=torch.bfloat16, device_map="cuda")  
    nvtx.pop_range(domain="nvtx-inference")
    
    nvtx.push_range("tokenize-prompt", domain="nvtx-inference")
    messages = [
        {
            "role": "system",
            "content": "You are a friendly chatbot who always responds in the style of a pirate. You response should be under 50 words.",
        },
        {"role": "user", "content": query},
    ]   
    prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    nvtx.pop_range(domain="nvtx-inference")
    
    nvtx.push_range("model inference", domain="nvtx-inference")
    outputs = pipe(prompt, max_new_tokens=150, do_sample=True, temperature=0.7, top_k=50,)
    print(outputs[0]["generated_text"])
    print("*********************************************************************")
    nvtx.pop_range(domain="nvtx-inference")
    nvtx.pop_range(domain="nvtx-inference")

invoke_model("How many helicopters can a human eat in one sitting? Can you also explain why? and tell a funny joke for 3 year old")

#invoke_model("Explain generative AI to a 6 year old student")
#invoke_model("why is the sky blue?")
#invoke_model("tell a unique funny joke to make a 3 year old laugh?")


## **Sample Profiling Reports**

![](../../static-assets/lab-01/llm-inference-optimized-rep-1.png)

![](../../static-assets/lab-01/llm-inference-optimized-rep-2.png)